In [1]:
# Imports
import torch
import torch.nn as nn
import torchvision

In [2]:
# Let's load the dataset
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

size = (64, 64)
transform = torchvision.transforms.Compose([torchvision.transforms.Resize(size), torchvision.transforms.ToTensor()])
train_dataset = list(torchvision.datasets.Flowers102("/tmp/flowers", "train", transform=transform, download=True))
test_dataset = list(torchvision.datasets.Flowers102("/tmp/flowers", "test", transform=transform, download=True))



In [3]:
train_images = torch.stack([img for img, _ in train_dataset], dim=0).to(device)
test_images = torch.stack([img for img, _ in test_dataset], dim=0).to(device)
train_labels = torch.tensor([label for _, label in train_dataset]).to(device)
test_labels = torch.tensor([label for _, label in test_dataset]).to(device)

# Let's make sure we only have two classes
train_images, train_labels = train_images[train_labels < 2], train_labels[train_labels < 2]
test_images, test_labels = test_images[test_labels < 2], test_labels[test_labels < 2]

In [ ]:
model = torch.nn.Linear(3 * 64 * 64, 1).to(device)
loss = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=.0001, momentum=0)

for epoch in range(100):
    #computes the output of the model (same as linear)
    out = model(train_images.view(-1, 3 * 64 * 64))

    #converts the output to a loss value (probability of being in class 1) aka binary regression
    loss_val = loss(out.squeeze(), train_labels.float())

    #compute gradients and update the weights
    optimizer.zero_grad()
    loss_val.backward()

    optimizer.step()
    print(f"{epoch=}: {loss_val.item()}")

    #results
    #starts just below .7 [which is 50/50 coin flip]
    #gets better to .5 after 100 epochs but that is still not good enough for a real model

epoch=0: 0.6784809827804565
epoch=1: 0.6762323379516602
epoch=2: 0.6739985346794128
epoch=3: 0.6717797517776489
epoch=4: 0.6695753335952759
epoch=5: 0.667385458946228
epoch=6: 0.6652098298072815
epoch=7: 0.663048267364502
epoch=8: 0.6609007120132446
epoch=9: 0.65876704454422
epoch=10: 0.6566471457481384
epoch=11: 0.6545407176017761
epoch=12: 0.6524478197097778
epoch=13: 0.6503682136535645
epoch=14: 0.648301899433136
epoch=15: 0.6462485790252686
epoch=16: 0.6442083716392517
epoch=17: 0.6421809196472168
epoch=18: 0.6401664614677429
epoch=19: 0.6381644010543823
epoch=20: 0.6361750364303589
epoch=21: 0.6341980695724487
epoch=22: 0.6322336792945862
epoch=23: 0.6302814483642578
epoch=24: 0.6283414363861084
epoch=25: 0.6264134049415588
epoch=26: 0.6244974136352539
epoch=27: 0.6225934028625488
epoch=28: 0.6207011342048645
epoch=29: 0.6188205480575562
epoch=30: 0.6169517040252686
epoch=31: 0.6150944232940674
epoch=32: 0.6132484674453735
epoch=33: 0.611413836479187
epoch=34: 0.6095906496047974
e

In [ ]:
def accuracy(pred: torch.Tensor, label: torch.Tensor) -> float:
    return ((pred > 0.5) == label).float().mean().item()


model = torch.nn.Linear(in_features=size[0] * size[1] * 3, out_features=1)
model = model.to(device)

loss_fn = nn.BCEWithLogitsLoss()
optim = torch.optim.SGD(model.parameters(), lr=2e-2)
num_epochs = 500

for epoch in range(num_epochs):
    pred = model(train_images.view(train_images.shape[0], -1))[..., 0]
    loss_val = loss_fn(pred, train_labels.float())

    optim.zero_grad()
    loss_val.backward()
    optim.step()

    if epoch % 25 == 0 or epoch == num_epochs - 1:
        print(f"{epoch =:5d}  loss = {loss_val.item():.2f}  accuracy(train) = {accuracy(pred, train_labels):.3f}")

    if epoch % 100 == 0 or epoch == num_epochs - 1:
        with torch.inference_mode():
            pred = model(test_images.view(test_images.shape[0], -1))[..., 0]
            print(f"   Accuracy (test): {accuracy(pred, test_labels):.3f}")